In [29]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,END
from langgraph.graph.message import add_messages
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict,Annotated

In [30]:
load_dotenv()

True

In [31]:
class MyState(TypedDict):
    receiver:str
    draft:str
    approval:str
    review_comments : str
    messages: Annotated[list,add_messages]

In [32]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [33]:
@tool
def send_email(draft:str)->dict:
    """This tool sends email"""
    print(draft)
    print("Email sent!")



In [34]:
llm_with_tools  = llm.bind_tools([send_email])

In [35]:
def llm_node(state:MyState)->dict:
    system = SystemMessage(content="You are a Helpful assitent.")
    response = llm_with_tools.invoke([system]+state["messages"])
    return {'messages':[response]}

def generate_email_draft(state:MyState)->dict:
    mes = state['messages']
    print("This is state:",state)
    if state.get('approval') is None:
        data = HumanMessage(content=f"reciver name is {state['receiver']}")
        draft = llm_with_tools.invoke(mes+[data])
    else:
        comments = HumanMessage(content=state['review_comments'])
        draft = llm_with_tools.invoke([comments]+mes)
    return {'draft':draft.content}


send_email_node = ToolNode([send_email])

def review_email(state:MyState)->dict:

    if state.get('draft') is not None:
        print("Review the email and provide approval (yes/no/edit) at the end.")
        print(state['draft'])
        res = input("Approval status:")

        if res.lower() == 'yes':
            return {'approval':'yes'}
        elif res.lower() == 'no':
            return {'approval':'no'}
        else:
            comments = input("Provide review comments")
            return{'approval':'edit','review_comments':comments}
    else:
        print('review called before drafting.')
        return {}
        



In [36]:
# define routing function
def after_review(state:MyState):
    last = state["messages"][-1]

    if getattr(last,"tool_calls") and state['approval'] == 'yes':
        return "send_email"
    elif state['approval'] == 'edit':
        return "draft"
    return "end"


In [37]:
flow = StateGraph(MyState)
flow.add_node("llm_node",llm_node)
flow.add_node("draft",generate_email_draft)
flow.add_node("review",review_email)
flow.add_node("send",send_email_node)

flow.set_entry_point("llm_node")

In [38]:
flow.add_edge("llm_node","draft")
flow.add_edge("draft","review")
flow.add_edge("review","send")
flow.add_conditional_edges("review",
                           after_review,
                           {
                               'send_email':'send',
                               'draft':'draft',
                               'end':END
                           }
                           )

In [39]:
memory = MemorySaver()

In [40]:
app = flow.compile(checkpointer=memory)
config1 = {'configurable':{'thread_id':'Narsimha_thread'}}
config2 = {'configurable':{'thread_id':'Anand_thread'}}

In [41]:
result = app.invoke({'messages':[HumanMessage(content="write a very short email to my friend.")],'receiver':'Narsimha'},
                    config = config1)

This is state: {'receiver': 'Narsimha', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='8bc4516f-aefe-419e-b0e0-33cb180f96bc'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say hi and see how you’re doing. Hope everything’s great on your end!\n\nCatch up soon?\n\nBest,  \n[Your Name]', additional_kwargs={'reasoning_content': 'User wants a very short email to their friend. Need to produce email content. Possibly ask for details? But they just say "write a very short email to my friend." So we can produce a generic short email. No need to send. Just write.'}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 137, 'total_tokens': 246, 'completion_time': 0.229503289, 'completion_tokens_details': {'reasoning_tokens': 53}, 'prompt_time': 0.005992276, 'prompt_tokens_details': None, 'queue_time': 0.052052234, 'total_time': 0.2354955

In [44]:
print(result)

{'receiver': 'Narsimha', 'draft': '**Subject:** Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say hi and see how you’re doing. By the way, I saw that red T‑shirt of yours—nice choice! I guess you’re trying to signal “stop” to all the boring conversations and keep things lively. 😄  \n\nHope everything’s great on your end! Let’s catch up soon.\n\nBest,  \n[Your Name]', 'approval': 'yes', 'review_comments': 'add a small joke about his red Tshirt', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='8bc4516f-aefe-419e-b0e0-33cb180f96bc'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say hi and see how you’re doing. Hope everything’s great on your end!\n\nCatch up soon?\n\nBest,  \n[Your Name]', additional_kwargs={'reasoning_content': 'User wants a very short email to their friend. Need to produce email content. Possibly ask for d

In [45]:
snapshot = app.get_state(config1)
print(snapshot)

StateSnapshot(values={'receiver': 'Narsimha', 'draft': '**Subject:** Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say hi and see how you’re doing. By the way, I saw that red T‑shirt of yours—nice choice! I guess you’re trying to signal “stop” to all the boring conversations and keep things lively. 😄  \n\nHope everything’s great on your end! Let’s catch up soon.\n\nBest,  \n[Your Name]', 'approval': 'yes', 'review_comments': 'add a small joke about his red Tshirt', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='8bc4516f-aefe-419e-b0e0-33cb180f96bc'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say hi and see how you’re doing. Hope everything’s great on your end!\n\nCatch up soon?\n\nBest,  \n[Your Name]', additional_kwargs={'reasoning_content': 'User wants a very short email to their friend. Need to produce email conten

Time Travel

In [46]:
snapshot = list(app.get_state_history(config1))
print(snapshot)

[StateSnapshot(values={'receiver': 'Narsimha', 'draft': '**Subject:** Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say hi and see how you’re doing. By the way, I saw that red T‑shirt of yours—nice choice! I guess you’re trying to signal “stop” to all the boring conversations and keep things lively. 😄  \n\nHope everything’s great on your end! Let’s catch up soon.\n\nBest,  \n[Your Name]', 'approval': 'yes', 'review_comments': 'add a small joke about his red Tshirt', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='8bc4516f-aefe-419e-b0e0-33cb180f96bc'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say hi and see how you’re doing. Hope everything’s great on your end!\n\nCatch up soon?\n\nBest,  \n[Your Name]', additional_kwargs={'reasoning_content': 'User wants a very short email to their friend. Need to produce email conte

In [48]:
time_travel = app.invoke(None,config = snapshot[-3].config)
print(time_travel)

This is state: {'receiver': 'Narsimha', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='8bc4516f-aefe-419e-b0e0-33cb180f96bc'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say hi and see how you’re doing. Hope everything’s great on your end!\n\nCatch up soon?\n\nBest,  \n[Your Name]', additional_kwargs={'reasoning_content': 'User wants a very short email to their friend. Need to produce email content. Possibly ask for details? But they just say "write a very short email to my friend." So we can produce a generic short email. No need to send. Just write.'}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 137, 'total_tokens': 246, 'completion_time': 0.229503289, 'completion_tokens_details': {'reasoning_tokens': 53}, 'prompt_time': 0.005992276, 'prompt_tokens_details': None, 'queue_time': 0.052052234, 'total_time': 0.2354955